<a href="https://colab.research.google.com/github/MukulGhai/UCS761-DEEP-LEARNING/blob/main/Lab_5_Digit_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
data = pd.read_csv("train.csv")

data = np.array(data)
np.random.shuffle(data)

Y = data[:,0]
X = data[:,1:]

In [ ]:
X = X / 255.0

In [ ]:
split = int(0.9 * X.shape[0])

X_train = X[:split]
Y_train = Y[:split]

X_val = X[split:]
Y_val = Y[split:]

In [ ]:
def one_hot(Y):
    one_hot_Y = np.zeros((Y.size,10))
    one_hot_Y[np.arange(Y.size),Y] = 1
    return one_hot_Y

Y_train_oh = one_hot(Y_train)

In [ ]:
np.random.seed(42)

W1 = np.random.randn(784,128) * 0.01
b1 = np.zeros((1,128))

W2 = np.random.randn(128,10) * 0.01
b2 = np.zeros((1,10))

In [ ]:
def relu(Z):
    return np.maximum(0,Z)

def relu_derivative(Z):
    return Z > 0

def softmax(Z):
    exp = np.exp(Z - np.max(Z,axis=1,keepdims=True))
    return exp / np.sum(exp,axis=1,keepdims=True)

In [ ]:
def forward(X):

    Z1 = X.dot(W1) + b1
    A1 = relu(Z1)

    Z2 = A1.dot(W2) + b2
    A2 = softmax(Z2)

    return Z1,A1,Z2,A2

In [ ]:
def backward(X,Y,Z1,A1,A2):

    m = X.shape[0]

    dZ2 = A2 - Y
    dW2 = A1.T.dot(dZ2) / m
    db2 = np.sum(dZ2,axis=0,keepdims=True) / m

    dZ1 = dZ2.dot(W2.T) * relu_derivative(Z1)
    dW1 = X.T.dot(dZ1) / m
    db1 = np.sum(dZ1,axis=0,keepdims=True) / m

    return dW1,db1,dW2,db2

In [ ]:
lr = 0.1
epochs = 50

for i in range(epochs):

    Z1,A1,Z2,A2 = forward(X_train)

    dW1,db1,dW2,db2 = backward(X_train,Y_train_oh,Z1,A1,A2)

    W1 -= lr * dW1
    b1 -= lr * db1

    W2 -= lr * dW2
    b2 -= lr * db2

    if i % 10 == 0:
        pred = np.argmax(A2,axis=1)
        acc = np.mean(pred == Y_train)
        print("Epoch:",i,"Accuracy:",acc)

Epoch: 0 Accuracy: 0.12452380952380952
Epoch: 10 Accuracy: 0.47185185185185186
Epoch: 20 Accuracy: 0.5149470899470899
Epoch: 30 Accuracy: 0.4671693121693122
Epoch: 40 Accuracy: 0.486005291005291


In [ ]:
_,_,_,A2_val = forward(X_val)

val_pred = np.argmax(A2_val,axis=1)

accuracy = np.mean(val_pred == Y_val)

print("Validation Accuracy:",accuracy)

Validation Accuracy: 0.5816666666666667


In [ ]:
print("First 20 Predictions:")
print(val_pred[:20])

print("Actual Labels:")
print(Y_val[:20])

First 20 Predictions:
[6 2 2 6 3 9 1 0 9 9 0 0 7 9 9 7 6 9 0 1]
Actual Labels:
[6 5 1 6 8 9 1 8 9 9 0 8 4 7 4 9 3 4 8 1]
